# 05 - Exportacao para PostgreSQL / DW

Objetivo deste notebook:
- conectar no PostgreSQL
- carregar dados tratados para `staging`
- preparar cargas para o schema `dw`
- validar o Data Mart usado pelo Metabase


## 1. Imports e configuracao


In [ ]:
import os
from sqlalchemy import create_engine, text

POSTGRES_HOST = os.getenv('POSTGRES_HOST', 'postgres-service')
POSTGRES_PORT = os.getenv('POSTGRES_PORT', '5432')
POSTGRES_DB = os.getenv('POSTGRES_DB', 'seguranca_publica')
POSTGRES_USER = os.getenv('POSTGRES_USER', 'postgres')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD', 'postgres')

DATABASE_URL = (
    f'postgresql+psycopg2://{POSTGRES_USER}:{POSTGRES_PASSWORD}'
    f'@{POSTGRES_HOST}:{POSTGRES_PORT}/{POSTGRES_DB}'
)


## 2. Testar conexao


In [ ]:
engine = create_engine(DATABASE_URL)

with engine.connect() as conn:
    resultado = conn.execute(text('SELECT current_database(), current_schema();')).fetchone()

resultado


## 3. Verificar schemas

Os schemas `raw`, `staging`, `dw` e `datamart_seguranca_publica` sao criados pelos scripts em `postgres-init/`.


In [ ]:
with engine.connect() as conn:
    schemas = conn.execute(text("""
        SELECT schema_name
        FROM information_schema.schemata
        WHERE schema_name IN ('raw', 'staging', 'dw', 'datamart_seguranca_publica')
        ORDER BY schema_name;
    """)).fetchall()

schemas


## 4. Proximas cargas

Depois que os notebooks 02 e 03 gerarem os dataframes tratados, esta etapa deve:
- gravar bases padronizadas em `staging`
- popular dimensoes faltantes em `dw`
- popular `dw.fact_municipio_ano`
- validar a view `datamart_seguranca_publica.vw_indicadores_municipio_ano`
